# VECTRI Advanced Analysis

This notebook contains advanced analysis of VECTRI model outputs, including seasonality, anomalies, and climate-disease lags.
**Data Sources:**
- Vector Data: `experment1.nc`
- Precipitation Data: `precip_processed.nc`
- Temperature Data: `temp_processed.nc`

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    CARTOPY_AVAILABLE = True
except ImportError:
    CARTOPY_AVAILABLE = False
    print("Cartopy not found, specific map projections will be disabled.")

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

In [ ]:
# Load Data
FILE_PATH = r'C:\Users\yonas\Documents\VECTRI-PYTHON\experment1.nc'
PRECIP_PATH = r'C:\Users\yonas\Documents\VECTRI-PYTHON\data\processed\precip_processed.nc'
TEMP_PATH = r'C:\Users\yonas\Documents\VECTRI-PYTHON\data\processed\temp_processed.nc'

try:
    # Load Precipitation First to get reference Time coordinates if needed
    ds_precip = xr.open_dataset(PRECIP_PATH)
    if 'latitude' in ds_precip.coords:
        ds_precip = ds_precip.rename({'latitude': 'lat', 'longitude': 'lon'}) 
        
    ds_temp = xr.open_dataset(TEMP_PATH)
    if 'latitude' in ds_temp.coords:
        ds_temp = ds_temp.rename({'latitude': 'lat', 'longitude': 'lon'}) 

    # Load Vector Data
    ds = xr.open_dataset(FILE_PATH, group="vector")
    print("Vector Data Loaded:", ds)
    
    # --- FIX START: Assign Coordinates if missing --- 
    if 'time' not in ds.coords:
        print("\'time\' coordinate missing in vector group. Attempting to assign from precipitation data...")
        if ds.sizes['time'] == ds_precip.sizes['time']:
            ds = ds.assign_coords(time=ds_precip.time)
            print("Assigned time coordinate from precip data.")
        else:
            print("Dimension mismatch, generating annual daily time index starting 1991-01-01.")
            # Assuming daily data for annual runs
            dates = pd.date_range(start='1991-01-01', periods=ds.sizes['time'], freq='D')
            ds = ds.assign_coords(time=dates)
            print("Created synthetic time index.")

    if 'latitude' in ds.coords:
        ds = ds.rename({'latitude': 'lat', 'longitude': 'lon'}) 
    elif 'lat' not in ds.coords and 'latitude' not in ds.coords:
         # If lat/lon dimensions exist but are not coords, try to copy from precip
         if ds.sizes['lat'] == ds_precip.sizes['lat'] and ds.sizes['lon'] == ds_precip.sizes['lon']:
             ds = ds.assign_coords(lat=ds_precip.lat, lon=ds_precip.lon)
             print("Assigned lat/lon coordinates from precip data.")
    # --- FIX END ---

except Exception as e:
    print(f"Error loading data: {e}")

## 1. Monthly Basis Analysis

In [ ]:
# 1. Monthly Climatology (Seasonality Profile)
vector_monthly_clim = ds['vector'].groupby('time.month').mean(dim='time')
vector_seasonality = vector_monthly_clim.mean(dim=['lat', 'lon'])

plt.figure(figsize=(10, 6))
vector_seasonality.plot(linewidth=2, marker='o', color='purple')
plt.title('Average Seasonal Cycle of Vector Density')
plt.ylabel('Vector Density (m^-2)')
plt.xlabel('Month')
plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.grid(True)
plt.show()

In [ ]:
# 2. Monthly Anomaly Analysis
vector_anomaly = ds['vector'].groupby('time.month') - vector_monthly_clim
vector_anomaly_ts = vector_anomaly.mean(dim=['lat', 'lon'])

plt.figure(figsize=(12, 6))
vector_anomaly_ts.plot(color='red', linewidth=1)
plt.title('Monthly Vector Density Anomaly (Deviation from Seasonal Mean)')
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.ylabel('Anomaly')
plt.show()

In [ ]:
# 3. Hovmöller Diagram (Latitude vs Time)
# Resample to monthly to reduce noise and file size influence
try:
    ds_monthly = ds['vector'].resample(time='1MS').mean()
    lat_time = ds_monthly.mean(dim='lon')

    plt.figure(figsize=(12, 8))
    lat_time.plot(x='time', y='lat', cmap='viridis')
    plt.title('Hovmöller Diagram: Vector Density (Latitude vs Time)')
    plt.ylabel('Latitude')
    plt.show()
except Exception as e:
    print(f"Could not plot Hovmoller: {e}")

## 2. Seasonal Basis Analysis

In [ ]:
# 4. Seasonal Spatial Maps
# Ensure seasons exist
try:
    ds_seasonal = ds['vector'].groupby('time.season').mean(dim='time')

    fig, axes = plt.subplots(2, 2, figsize=(16, 12), 
                             subplot_kw={'projection': ccrs.PlateCarree()} if CARTOPY_AVAILABLE else {})
    seasons = ['DJF', 'MAM', 'JJA', 'SON']

    for i, season in enumerate(seasons):
        ax = axes.flat[i]
        if CARTOPY_AVAILABLE:
            ax.coastlines()
            ax.add_feature(cfeature.BORDERS, linestyle=':')
        
        if season in ds_seasonal.season:
            data = ds_seasonal.sel(season=season)
            if CARTOPY_AVAILABLE:
                data.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='YlGnBu', cbar_kwargs={'label': 'Vector Density'})
            else:
                data.plot(ax=ax, cmap='YlGnBu')
            ax.set_title(f'Seasonal Mean: {season}')
        else:
            ax.text(0.5, 0.5, 'Season not present', ha='center')

    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Error plotting seasonal maps: {e}")

In [ ]:
# 5. Transmission Season Length
# Define threshold (e.g., mean value or specific density)
threshold = ds['vector'].mean().values
# Count months per year where density > threshold
monthly_counts = (ds['vector'].resample(time='1MS').mean() > threshold).groupby('time.year').sum(dim='time')
avg_season_length = monthly_counts.mean(dim='year')

plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree()) if CARTOPY_AVAILABLE else plt.axes()
if CARTOPY_AVAILABLE: ax.coastlines()
p = avg_season_length.plot(ax=ax, transform=ccrs.PlateCarree() if CARTOPY_AVAILABLE else None, 
                       cmap='RdYlBu_r', cbar_kwargs={'label': 'Months per Year'})
plt.title('Average Transmission Season Length (Months/Year)')
plt.show()

In [ ]:
# 6. Peak Timing Map
# Month index of peak transmission
peak_month = ds['vector'].groupby('time.month').mean(dim='time').idxmax(dim='month')

plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree()) if CARTOPY_AVAILABLE else plt.axes()
if CARTOPY_AVAILABLE: ax.coastlines()
peak_month.plot(ax=ax, transform=ccrs.PlateCarree() if CARTOPY_AVAILABLE else None, 
                cmap='twilight', levels=13, cbar_kwargs={'label': 'Month'})
plt.title('Month of Peak Transmission')
plt.show()

## 3. Annual Basis Analysis

In [ ]:
# 7. Inter-Annual Variability (Trend)
annual_mean = ds['vector'].resample(time='1YS').mean().mean(dim=['lat', 'lon'])

plt.figure(figsize=(12, 5))
annual_mean.plot(marker='o', linestyle='-', color='green')
plt.title('Inter-Annual Variability of Vector Density (Domain Average)')
plt.ylabel('Mean Vector Density')
plt.grid(True)
plt.show()

In [ ]:
# 8. Year-to-Year Spatial Comparison (Early vs Late)
start_year = int(ds.time.dt.year.min())
end_year = int(ds.time.dt.year.max())

# Mean of first 5 years vs last 5 years
early_period = ds['vector'].sel(time=slice(f'{start_year}-01-01', f'{start_year+4}-12-31')).mean(dim='time')
late_period = ds['vector'].sel(time=slice(f'{end_year-4}-01-01', f'{end_year}-12-31')).mean(dim='time')
diff = late_period - early_period

plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree()) if CARTOPY_AVAILABLE else plt.axes()
if CARTOPY_AVAILABLE: ax.coastlines()
diff.plot(ax=ax, transform=ccrs.PlateCarree() if CARTOPY_AVAILABLE else None, 
          cmap='RdBu_r', cbar_kwargs={'label': 'Difference (Late - Early)'})
plt.title(f'Change in Vector Density: {start_year}-{start_year+4} vs {end_year-4}-{end_year}')
plt.show()

## 4. Statistical & Advanced Analysis

In [ ]:
# 9. Climate-Disease Lag Analysis
# Determine precipitation variable name
precip_var = 'tp' if 'tp' in ds_precip else list(ds_precip.data_vars)[0]
print(f"Using precipitation variable: {precip_var}")

# Spatially aggregate
precip_ts = ds_precip[precip_var].mean(dim=['lat', 'lon']).resample(time='1MS').mean()
vector_ts = ds['vector'].mean(dim=['lat', 'lon']).resample(time='1MS').mean()

# Align time series
common_time = np.intersect1d(precip_ts.time, vector_ts.time)
p_aligned = precip_ts.sel(time=common_time)
v_aligned = vector_ts.sel(time=common_time)

# Calculate cross-correlation
lags = range(0, 7)
corrs = []
for lag in lags:
    # Shift precip forward by lag months (P affects V in future)
    # corr(P(t-lag), V(t))
    corr = np.corrcoef(p_aligned.shift(time=lag).dropna(dim='time'), v_aligned.dropna(dim='time'))[0, 1]
    corrs.append(corr)

plt.figure(figsize=(8, 5))
plt.bar(lags, corrs, color='skyblue')
plt.xlabel('Lag (Months)')
plt.ylabel('Correlation Coefficient')
plt.title('Lagged Correlation: Precipitation vs Vector Density')
plt.grid(axis='y')
plt.show()

In [ ]:
# 10. Exceedance Probability
# Probability of exceeding the 75th percentile of vector density
thresh_75 = ds['vector'].quantile(0.75).values
prob_exceed = (ds['vector'] > thresh_75).mean(dim='time')

plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree()) if CARTOPY_AVAILABLE else plt.axes()
if CARTOPY_AVAILABLE: ax.coastlines()
prob_exceed.plot(ax=ax, transform=ccrs.PlateCarree() if CARTOPY_AVAILABLE else None, 
                 cmap='Reds', cbar_kwargs={'label': 'Probability'})
plt.title(f'Probability of Exceeding High Transmission Threshold (> {thresh_75:.2f})')
plt.show()